In [ ]:
import os
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

INPUT_PATH  = "../../../data/phase2/features_for_model.parquet"
OUTPUT_PATH = "../../../data/phase2/validation_results.parquet"

FEATURE_COLS = [
    "ema_ratio", "rsi_14", "macd_hist", "atr_14",
    "session_quality_enc", "direction_enc", "signal_valid_enc",
]
LABEL_COL = "label"

In [ ]:
def train_fold(
    X_train: pd.DataFrame,
    y_train: pd.Series,
    X_test: pd.DataFrame,
) -> tuple[np.ndarray, StandardScaler, LogisticRegression]:
    """
    Fit StandardScaler + LogisticRegression on training data.
    Return (predicted_probabilities_for_test, fitted_scaler, fitted_model).

    Scaler is fit only on X_train to prevent data leakage.
    LogisticRegression uses C=1.0, max_iter=1000, solver='lbfgs'.
    Returns probabilities for the positive class (label=1).
    """
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled  = scaler.transform(X_test)

    model = LogisticRegression(C=1.0, max_iter=1000, solver="lbfgs", random_state=42)
    model.fit(X_train_scaled, y_train)

    proba = model.predict_proba(X_test_scaled)[:, 1]  # P(label=1)
    return proba, scaler, model


def run_walk_forward(df: pd.DataFrame) -> pd.DataFrame:
    """
    Run walk-forward validation across all folds.

    For each fold N:
      1. Filter train rows (fold==N, split=="train")
      2. Filter test rows  (fold==N, split=="test")
      3. Call train_fold
      4. Append test rows + predicted probability to results

    Returns DataFrame with columns:
      date, s3_key, fold, label, prob_itm, signal_valid_enc, direction_enc
    """
    folds = sorted(df[df["split"] == "test"]["fold"].unique())
    results = []

    for fold_idx in folds:
        train = df[(df["fold"] == fold_idx) & (df["split"] == "train")]
        test  = df[(df["fold"] == fold_idx) & (df["split"] == "test")]

        if len(train) < 10:
            print(f"Fold {fold_idx}: skipping — only {len(train)} train rows")
            continue
        if len(test) == 0:
            print(f"Fold {fold_idx}: skipping — no test rows")
            continue

        X_train = train[FEATURE_COLS]
        y_train = train[LABEL_COL]
        X_test  = test[FEATURE_COLS]

        proba, _, model = train_fold(X_train, y_train, X_test)

        fold_results = test[["date", "s3_key", "fold", LABEL_COL, "signal_valid_enc", "direction_enc"]].copy()
        fold_results["prob_itm"] = proba

        n_pos = int((y_train == 1).sum())
        n_neg = int((y_train == 0).sum())
        print(f"Fold {fold_idx}: train={len(train)} (pos={n_pos}, neg={n_neg}), test={len(test)}, coef={model.coef_[0].round(3)}")
        results.append(fold_results)

    if not results:
        raise RuntimeError("No folds produced results.")
    return pd.concat(results, ignore_index=True)

In [ ]:
df = pd.read_parquet(INPUT_PATH)
results = run_walk_forward(df)

print(f"\nTotal test predictions: {len(results)}")
print(f"prob_itm range: {results['prob_itm'].min():.3f} – {results['prob_itm'].max():.3f}")
print(f"Label distribution in test set:\n{results['label'].value_counts()}")

os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)
results.to_parquet(OUTPUT_PATH, index=False)
print(f"\nSaved to {OUTPUT_PATH}")

In [ ]:
df = pd.read_parquet(OUTPUT_PATH)
print(df[["date", "s3_key", "fold", "label", "prob_itm"]].head(20).to_string())